# 03 — Learning rate, rounds, and overfitting behaviour

*Chapter 07 — AdaBoost · Notebook 3 of 5*

Notebook 2 ended on a worry. The additive model drove the **training** error toward zero — and
everything we learned in module 00 says a model flexible enough to memorize its training set should
**overfit**. Does AdaBoost? The answer is famously two-faced, and it is shaped by a single knob, the
**learning rate**. This notebook turns that knob and watches how the ensemble generalizes: a happy
surprise on clean data, an honest limit on noisy data.

**Prerequisites.**
- **Notebooks 1–2:** the reweighting loop, the learner weight α, the additive model F = Σ αₜ hₜ, and
  the exponential loss that punishes a confidently-wrong point without bound.
- **Module 00:** over- and under-fitting, the train/test error U-curve, the generalization gap.
- **Chapter 06:** a random forest's diminishing returns as trees are added — a useful contrast.

**What you'll be able to do.**
- Use `learning_rate` (shrinkage) and read how it trades off against the number of rounds.
- Explain AdaBoost's **resistance to overfitting** on clean data — and the margin idea behind it.
- Recognize and diagnose its **overfitting under label noise**, and reach for early stopping.

## Where we are — one concept: controlling complexity

A boosted model's complexity grows with the number of **rounds**: each weak learner adds another term.
The **learning rate** shrinks how much each term counts. Those two dials — how many rounds, and how big
each step — are how boosting controls how hard it fits the data.

This notebook follows a single thread: turn those dials and watch generalization. First on clean data,
where AdaBoost does something module 00 would not predict; then on noisy data, where it does exactly
what module 00 warns about. Same model, two faces.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0

# The same two crescents as notebooks 1-2.
X, y = make_moons(n_samples=400, noise=0.20, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
print(f"train: {X_train.shape[0]} points    test: {X_test.shape[0]} points")

## The learning rate (shrinkage)

AdaBoost lets us multiply every learner's contribution by a factor ν between 0 and 1, the
**learning rate**:

$$F(x) = \sum_{t} \nu\, \alpha_t\, h_t(x).$$

The same ν also scales the reweighting step. It is a **brake**: with ν = 1 each round counts in full;
with ν = 0.1 each round counts a tenth as much, so you need far more rounds to build the same model.
If that feels familiar, it is the role the step size played in chapter 03's gradient descent — small
steps are slower but often land more gently. Let us confirm ν does exactly this.

In [ ]:
# Round 1's weighted error (uniform weights) gives alpha = ln((1-eps)/eps).
w0 = np.full(len(y_train), 1.0 / len(y_train))
stump = DecisionTreeClassifier(max_depth=1, random_state=SEED)
stump.fit(X_train, y_train, sample_weight=w0)
eps1 = w0[stump.predict(X_train) != y_train].sum()
alpha1 = np.log((1 - eps1) / eps1)
print(f"round-1 weighted error eps = {eps1:.4f},  alpha = ln((1-eps)/eps) = {alpha1:.4f}\n")

# learning_rate scales that alpha by nu. Check against sklearn's estimator_weights_.
for nu in (1.0, 0.5, 0.1):
    a = AdaBoostClassifier(n_estimators=3, learning_rate=nu, random_state=SEED)
    a.fit(X_train, y_train)
    print(f"lr={nu}:  nu * alpha = {nu * alpha1:.4f}    "
          f"estimator_weights_[0] = {a.estimator_weights_[0]:.4f}")

**Read the result.** `learning_rate` does one thing: it scales every vote weight α — and the
matching reweighting step — by ν. scikit-learn's `estimator_weights_` confirms it: at ν = 0.5 the first
learner's weight is half of 1.68, at ν = 0.1 it is a tenth. Smaller steps move the ensemble less each
round, so reaching a given fit takes more rounds. That sets up the trade-off we measure shortly — but
first, the clean-data surprise.

## Does driving training error to zero overfit?

Module 00 taught the U-curve: as a model grows more flexible, training error keeps falling but test
error eventually turns back **up** — the model starts memorizing noise instead of learning signal.
Notebook 2 showed AdaBoost drives **training** error to zero by about 114 rounds. By the U-curve story
we should expect test error to rise well before that. Let us measure both, round by round.

In [ ]:
clean = AdaBoostClassifier(n_estimators=400, learning_rate=1.0, random_state=SEED)
clean.fit(X_train, y_train)
rounds = np.arange(1, 401)
train_err = np.array([1 - s for s in clean.staged_score(X_train, y_train)])
test_err = np.array([1 - s for s in clean.staged_score(X_test, y_test)])
first0 = int(np.argmax(train_err == 0)) + 1

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rounds, train_err, color=COLORS["train"], linewidth=2.0, label="training error")
ax.plot(rounds, test_err, color=COLORS["test"], linewidth=2.0, label="test error")
ax.axvline(first0, color=COLORS["muted"], linestyle="--", linewidth=1.2,
           label=f"training error hits 0 (round {first0})")
ax.set_xlabel("number of rounds")
ax.set_ylabel("error")
ax.set_ylim(0, 0.25)
ax.legend(loc="upper right")
plt.show()
print(f"train error 0 from round {first0};  test min {test_err.min():.4f} "
      f"@ round {test_err.argmin() + 1};  test @ 400 = {test_err[-1]:.4f}")

**Read the figure.** Two things happen, and the second is the surprise. Training error falls to
**zero around round 114** (bar one transient blip) and holds there — the ensemble has memorized every
training point. But the **test** error does *not* turn up: it bottoms near **0.042** around round 35
and then holds inside a narrow **0.04–0.06** band all the way out to 400 rounds (a mild drift of about
0.017, not a climb). Adding rounds long after the training error is zero costs us almost nothing.

This is AdaBoost's celebrated **resistance to overfitting**, and there is an explanation (Schapire,
Freund, Bartlett & Lee, 1998). Once every training point is on the correct side, extra rounds keep
pushing the **hardest** points further from the boundary — widening the smallest **margins**, the
confidence with which the closest points are classified. Larger margins tend to generalize better, so
test error keeps from rising even as the model grows. The honest word is **resistance**, not immunity —
and the next experiment shows why that distinction matters.

## The trade-off: rounds versus step size

If a smaller learning rate takes smaller steps, it should need more rounds to reach the same place.
Let us race three learning rates on the same clean data.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
lr_colors = {1.0: COLORS["model"], 0.5: COLORS["highlight"], 0.1: COLORS["correct"]}
for nu, color in lr_colors.items():
    a = AdaBoostClassifier(n_estimators=400, learning_rate=nu, random_state=SEED)
    a.fit(X_train, y_train)
    te = np.array([1 - s for s in a.staged_score(X_test, y_test)])
    ax.plot(np.arange(1, 401), te, color=color, linewidth=2.0, label=f"learning_rate = {nu}")
ax.set_xlabel("number of rounds")
ax.set_ylabel("test error")
ax.set_ylim(0, 0.25)
ax.legend(loc="upper right")
plt.show()

**Read the figure.** The full-step model (ν = 1.0) reaches its plateau of about 0.058 within
roughly **10 rounds**. The half-step (ν = 0.5) settles more slowly but holds the **lowest** band
through the middle rounds (~0.05, below the full-step plateau). The gentle ν = 0.1 **crawls** — it
needs on the order of **400 rounds** even to reach that level. That is the trade-off: a smaller
learning rate is slower but can generalize slightly better, so the two knobs are tuned **together**
(you will do exactly that with `GridSearchCV` in notebook 4). More rounds is not free — and, as the
next experiment shows, not always safe.

## The other face: label noise

The resistance we saw held on **clean** data. But recall what AdaBoost does: it chases the points it
keeps getting wrong (notebook 1), and the exponential loss punishes a confidently-wrong point without
bound (notebook 2). Now imagine a point whose label is **wrong** — mislabeled in the data. It can never
be classified "correctly", so it looks eternally hard, and the reweighting multiplies its weight by
exp(α) every round it stays wrong — feeding it more and more. On noisy data, then, more rounds should
*hurt*. Let us flip a quarter of the
**training** labels — leaving the test set clean and honest — and watch.

In [ ]:
# Flip 25% of the TRAINING labels; the test set stays clean and honest.
rng = np.random.default_rng(SEED)
y_noisy = y_train.copy()
flip_idx = rng.choice(len(y_noisy), size=int(0.25 * len(y_noisy)), replace=False)
y_noisy[flip_idx] = 1 - y_noisy[flip_idx]

noisy = AdaBoostClassifier(n_estimators=400, learning_rate=1.0, random_state=SEED)
noisy.fit(X_train, y_noisy)
te_n = np.array([1 - s for s in noisy.staged_score(X_test, y_test)])  # test stays clean
stop = int(te_n.argmin()) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Left: clean vs noisy training, both scored on the SAME clean test set (comparable).
axes[0].plot(rounds, test_err, color=COLORS["correct"], linewidth=2.0,
             label="clean training data")
axes[0].plot(rounds, te_n, color=COLORS["error"], linewidth=2.0,
             label="25% of train labels flipped")
axes[0].axvline(stop, color=COLORS["muted"], linestyle="--", linewidth=1.2,
                label=f"noisy test bottoms (round {stop})")
axes[0].set_xlabel("number of rounds")
axes[0].set_ylabel("test error (clean test set)")
axes[0].set_ylim(0, 0.30)
axes[0].legend(loc="upper left")
axes[0].set_title("more rounds: flat on clean data, harmful under noise")

# Right: the noisy-trained boundary at 400 rounds, with the flipped points ringed.
viz.plot_decision_boundary(noisy, X_train, y_train, ax=axes[1])
axes[1].scatter(X_train[flip_idx, 0], X_train[flip_idx, 1], s=150, facecolors="none",
                edgecolors=COLORS["error"], linewidth=1.8, label="flipped labels")
axes[1].legend(loc="best")
axes[1].set_title("boundary at 400 rounds: islands around the noise")
plt.show()
print(f"clean test @ 400 = {test_err[-1]:.4f};  noisy test min {te_n.min():.4f} @ round {stop}, "
      f"@ 400 = {te_n[-1]:.4f}")

**Read the figure.** The left panel puts the two worlds side by side, both measured on the same
clean test set. With **clean** training data (teal) the test error stays flat near 0.058 however many
rounds you add — the resistance from Figure A. Flip a **quarter** of the training labels (coral) and the
story reverses: the test error bottoms early, around **round 18** at ~0.067, then **climbs to ~0.150** —
past that point, every extra round makes the model *worse*. The right panel shows why: the boundary no
longer tracks the two crescents cleanly — it carves little **islands** around the ringed flipped points,
contorting itself to fit labels that are wrong. This is AdaBoost's well-known weak spot (Dietterich,
2000): the exponential loss pours ever more weight onto the impossible points, so adding rounds fits
more and more noise. From its bottom to round 400 the clean test error drifts up only ~0.017; the noisy
one climbs ~0.083 — about five times as far. **Resistance was never immunity.**

## What to do about it

The noisy curve hands us the fix. Three honest levers:

- **Early stopping.** The test error bottomed around round 18, then rose. Choosing the number of rounds
  by held-out (or cross-validated) error — rather than running to the end — recovers most of the loss.
  This barely mattered on clean data, where the curve was flat; on noisy data it is essential.
- **A smaller learning rate.** Gentler steps overfit the noise more slowly, buying a wider, safer
  plateau.
- **A simpler base learner** (notebook 4's `estimator`), or a more **robust loss** (gradient boosting,
  chapter 08), or bagging's averaging instead of boosting's chasing (chapter 06). On genuinely noisy
  problems one of these often beats AdaBoost outright — the capstone (notebook 5) measures it on real
  data.

## Honest scoping

- **"Resistance" is empirical, not a guarantee.** It is the margin behaviour Schapire et al. (1998)
  described, and it holds at low noise; it is not a theorem that AdaBoost cannot overfit.
- **The noise overfit is real and measured** (Dietterich, 2000) — the second figure is not a quirk of a
  single run.
- **These exact rounds are this dataset, this seed** (moons with noise 0.20, seed 0). The *shape* of
  both behaviours generalizes; the precise round numbers do not.
- **No leakage.** One clean, sealed test set throughout; the label noise was injected into the
  **training** set only, which is where real mislabeling lives.

## Your turn

1. **Read the curves (easy).** From the noisy figure, at roughly what round does the test error reach
   its minimum, and what does it do afterwards? Contrast that in one sentence with the clean curve in
   Figure A.
2. **Turn up the noise (medium).** Re-run the noise experiment at `flip = 0.10` and `flip = 0.40`. Does
   the overfit get worse as noise rises? Does the test minimum arrive *earlier* with more noise?
3. **Early stopping (harder).** On the noisy run, find the round with the lowest test error and read off
   that error; compare to the test error at 400 rounds (accuracy = 1 − error). Then explain why early
   stopping rescues the noisy model but barely changes the clean one — point to what each model is
   fitting in its later rounds.

## What you built

- You used **`learning_rate`** (shrinkage) and confirmed it scales every vote weight by ν — a brake
  that trades **more rounds** for **smaller steps**.
- You measured AdaBoost's **resistance to overfitting** on clean data: training error hit zero at round
  114, yet test error held in a narrow band — the **margin** idea (Schapire et al., 1998).
- You measured the other face — **overfitting under label noise**: with a quarter of the training labels
  flipped, test error bottomed near round 18 and then climbed, the boundary carving islands around the
  mislabeled points.
- You read the fix: **early stopping**, a smaller learning rate, a simpler base learner — and saw why it
  matters under noise but not on clean data.

**Vocabulary you now own:** learning rate / shrinkage · the rounds × learning-rate trade-off ·
resistance to overfitting (margins) · overfitting under label noise · early stopping.

## References

- Schapire, R. E., Freund, Y., Bartlett, P., & Lee, W. S. (1998). *Boosting the margin: a new
  explanation for the effectiveness of voting methods.* The Annals of Statistics 26(5), 1651–1686.
  DOI: [10.1214/aos/1024691352](https://doi.org/10.1214/aos/1024691352)
- Dietterich, T. G. (2000). *An experimental comparison of three methods for constructing ensembles of
  decision trees: bagging, boosting, and randomization.* Machine Learning 40(2), 139–157.
  DOI: [10.1023/A:1007607513941](https://doi.org/10.1023/A:1007607513941)
- Freund, Y., & Schapire, R. E. (1997). *A decision-theoretic generalization of on-line learning and an
  application to boosting.* Journal of Computer and System Sciences 55(1), 119–139.
  DOI: [10.1006/jcss.1997.1504](https://doi.org/10.1006/jcss.1997.1504)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10.4–10.6 (shrinkage and boosting).
  DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)

*Previous: 02 — Weak learners and the additive model. Next: 04 — The estimator `AdaBoostClassifier`
and its parameters.*